# MINI PROJECT 07 ONLINE RETAIL II

In [1]:
# IMPORT
import pandas as pd
import numpy as np

C:\Users\VAIBHAV\AppData\Local\Temp\ipykernel_20428\89779171.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# LOAD DATASET
df = pd.read_excel("online_retail_II.xlsx")
df.shape

(525461, 8)

In [3]:
# BASIC INSPECTION
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.info

<bound method DataFrame.info of        Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
525456  538171     22271                 FELTCRAFT DOLL ROSIE         2   
525457  538171     22750         FELTCRAFT PRINCESS LOLA DOLL         1   
525458  538171     22751       FELTCRAFT PRINCESS OLIVIA DOLL         1   
525459  538171     20970   PINK FLORAL FELTCRAFT SHOULDER BAG         2   
525460  538171     21931               JUMBO STORAGE BAG SUKI         2   

               InvoiceDate  Price  Customer ID         Country  
0 

In [5]:
# MISSING VALUES
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [6]:
# DUPLICATE ROWS
df.duplicated().sum()

6865

In [7]:
# COLUMN NAMES
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [8]:
# CHECK DATATYPES
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

### Data Cleaning

In [9]:
df = df.drop_duplicates()
df.shape

(518596, 8)

In [10]:
# Check missing description
df[df["Description"].isnull()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [11]:
# We'll remove rows where the product description is missing because product-level analysis requires a valid product description.
df = df.dropna(subset=["Description"])
df.shape

(515668, 8)

In [12]:
# Verify Cleaning
df.duplicated().sum()

0

In [13]:
df.isnull().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    104905
Country             0
dtype: int64

In [14]:
# Core Business Matrix
df["Sales"] = df["Quantity"] * df["Price"]
df[[
     "Quantity" , "Price" , "Sales"
]].head(10)

,Quantity,Price,Sales
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0
5,24,1.65,39.6
6,24,1.25,30.0
7,10,5.95,59.5
8,12,2.55,30.6
9,12,3.75,45.0


### CUSTOMER ANALYSIS

In [15]:
customer_df = df.dropna(subset=["Customer ID"]).copy()

In [16]:
# customer KPI
customer_analysis = customer_df.groupby("Customer ID").agg(
     Total_Sales = ("Sales" , "sum"),
     Total_Orders = ("Invoice" , "nunique"),
     Total_Quantity = ("Quantity" , "sum")
)
customer_analysis.head()

,Total_Sales,Total_Orders,Total_Quantity
Customer ID,,,
12346.0,-51.74,15,53
12347.0,1323.32,2,828
12348.0,222.16,1,373
12349.0,2646.99,4,988
12351.0,300.93,1,261


In [17]:
# customer contribution
total_Sales = customer_analysis["Total_Sales"].sum()

customer_analysis["Sales_Contribution_%"] = (
     customer_analysis["Total_Sales"]
     .div(total_Sales).mul(100)
)

In [18]:
top_customers = customer_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
).head(10)
top_customers

,Total_Sales,Total_Orders,Total_Quantity,Sales_Contribution_%
Customer ID,,,,
18102.0,341776.73,95,122988,4.110435
14646.0,243853.05,87,168501,2.932739
14156.0,183163.55,138,106883,2.202847
14911.0,137710.76,270,66572,1.656201
13694.0,128172.42,105,123639,1.541487
17511.0,83760.60,42,54901,1.007361
15061.0,82162.74,90,51395,0.988144
16684.0,75610.17,34,51705,0.909338
13089.0,55801.73,132,28422,0.671109


### PRODUCT ANALYSIS

In [19]:
product_analysis = df.groupby(
    ["StockCode", "Description"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Total_Invoices=("Invoice", "nunique")
)

product_analysis.head()

Total_Sales  Total_Quantity  \
StockCode Description                                                
10002     INFLATABLE POLITICAL GLOBE       5450.21            6748   
10080     GROOVY CACTUS INFLATABLE           10.20              12   
10109     BENDY COLOUR PENCILS                1.68               4   
10120     DOGGY RUBBER                       99.27             457   
          Zebra invcing error                 0.00           -9000   

                                       Total_Invoices  
StockCode Description                                  
10002     INFLATABLE POLITICAL GLOBE              317  
10080     GROOVY CACTUS INFLATABLE                  5  
10109     BENDY COLOUR PENCILS                      1  
10120     DOGGY RUBBER                             48  
          Zebra invcing error                       1

In [20]:
# Top 10 products by revenue
top_products = product_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
)
top_products.head(10)

,,Total_Sales,Total_Quantity,Total_Invoices
StockCode,Description,,,
22423,REGENCY CAKESTAND 3 TIER,162885.71,13080,2186
85123A,WHITE HANGING HEART T-LIGHT HOLDER,155540.37,57327,3375
DOT,DOTCOM POSTAGE,116401.99,731,735
84879,ASSORTED COLOUR BIRD ORNAMENT,72251.32,44805,1420
22086,PAPER CHAIN KIT 50'S CHRISTMAS,57808.25,17062,964
85099B,JUMBO BAG RED RETROSPOT,54217.92,30249,1253
47566,PARTY BUNTING,49626.92,10084,1017
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47641.89,13663,351
POST,POSTAGE,46092.36,2154,862


In [21]:
# Product contribution
product_analysis["Sales_Contribution_%"] = (
    product_analysis["Total_Sales"]
    .div(product_analysis["Total_Sales"].sum())
    .mul(100)
)

In [22]:
top_products = product_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
)
top_products.head(10)

,,Total_Sales,Total_Quantity,Total_Invoices,Sales_Contribution_%
StockCode,Description,,,,
22423,REGENCY CAKESTAND 3 TIER,162885.71,13080,2186,1.713545
85123A,WHITE HANGING HEART T-LIGHT HOLDER,155540.37,57327,3375,1.636272
DOT,DOTCOM POSTAGE,116401.99,731,735,1.224540
84879,ASSORTED COLOUR BIRD ORNAMENT,72251.32,44805,1420,0.760078
22086,PAPER CHAIN KIT 50'S CHRISTMAS,57808.25,17062,964,0.608138
85099B,JUMBO BAG RED RETROSPOT,54217.92,30249,1253,0.570368
47566,PARTY BUNTING,49626.92,10084,1017,0.522071
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47641.89,13663,351,0.501189
POST,POSTAGE,46092.36,2154,862,0.484888


## Advanced Pandas Business Analysis

### Customer Contribution Within Country

In [24]:
country_sales = df.groupby(
    "Country"
)["Sales"].transform("sum")

df["Country_Total_Sales"] = country_sales

In [26]:
df["Country_Sales_Contribution_%"] = (
    df["Sales"]
    .div(df["Country_Total_Sales"])
    .mul(100)
)
df[[
     "Country",
     "Sales" ,
     "Country_Total_Sales" ,
     "Country_Sales_Contribution_%"
]].head(10)

,Country,Sales,Country_Total_Sales,Country_Sales_Contribution_%
0,United Kingdom,83.4,8161692.763,0.001022
1,United Kingdom,81.0,8161692.763,0.000992
2,United Kingdom,81.0,8161692.763,0.000992
3,United Kingdom,100.8,8161692.763,0.001235
4,United Kingdom,30.0,8161692.763,0.000368
5,United Kingdom,39.6,8161692.763,0.000485
6,United Kingdom,30.0,8161692.763,0.000368
7,United Kingdom,59.5,8161692.763,0.000729
8,United Kingdom,30.6,8161692.763,0.000375
9,United Kingdom,45.0,8161692.763,0.000551


### Prduct Ranking

In [27]:
# product level sales
product_rank = df.groupby(
     ["Country" , "Description"]
).agg(
     Total_Sales = ("Sales" , "sum")
).reset_index()

In [28]:
product_rank["Product_Rank"] = (
    product_rank
    .groupby("Country")["Total_Sales"]
    .rank(
        method="dense",
        ascending=False
    )
)

In [29]:
top_country_products = product_rank[
    product_rank["Product_Rank"] <= 3
]

top_country_products.sort_values(
    ["Country", "Product_Rank"]
).head(20)

,Country,Description,Total_Sales,Product_Rank
325,Australia,REGENCY CAKESTAND 3 TIER,927.00,1.0
312,Australia,RED RETROSPOT ROUND CAKE TINS,546.95,2.0
383,Australia,SET/2 RED RETROSPOT TEA TOWELS,510.00,3.0
695,Austria,POSTAGE,1520.00,1.0
565,Austria,EDWARDIAN PARASOL NATURAL,439.80,2.0
564,Austria,EDWARDIAN PARASOL BLACK,368.40,3.0
567,Austria,EDWARDIAN PARASOL RED,368.40,3.0
921,Bahrain,WHITE TALL PORCELAIN T-LIGHT HOLDER,202.50,1.0
865,Bahrain,KENSINGTON COFFEE SET,127.50,2.0
882,Bahrain,PICCADILLY TEA SET,119.60,3.0


### Monthly Sales + shift()

In [30]:
# create monthly sales
monthly_sales = df.groupby(
    [
        df["InvoiceDate"].dt.to_period("M"),
        "Country"
    ]
)["Sales"].sum().reset_index()

monthly_sales.columns = [
    "Month",
    "Country",
    "Total_Sales"
]

In [31]:
monthly_sales = monthly_sales.sort_values(
    ["Country", "Month"]
)

In [32]:
monthly_sales["Previous_Month_Sales"] = (
    monthly_sales
    .groupby("Country")["Total_Sales"]
    .shift(1)
)

In [33]:
monthly_sales["MoM_Change_%"] = (
    monthly_sales["Total_Sales"]
    .sub(monthly_sales["Previous_Month_Sales"])
    .div(monthly_sales["Previous_Month_Sales"])
    .mul(100)
)

In [34]:
monthly_sales.head(15)

,Month,Country,Total_Sales,Previous_Month_Sales,MoM_Change_%
0,2009-12,Australia,50.25,NaN,NaN
43,2010-02,Australia,1029.66,50.25,1949.074627
62,2010-03,Australia,429.39,1029.66,-58.297885
85,2010-04,Australia,630.95,429.39,46.941009
108,2010-05,Australia,1312.70,630.95,108.051351
128,2010-06,Australia,3189.28,1312.70,142.955740
152,2010-07,Australia,633.62,3189.28,-80.132820
173,2010-08,Australia,176.00,633.62,-72.223099
196,2010-09,Australia,785.83,176.00,346.494318
224,2010-10,Australia,2989.15,785.83,280.381253


## Final Business Analysis

### Country Performance

In [35]:
country_analysis = df.groupby("Country").agg(
    Total_Sales=("Sales", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Total_Invoices=("Invoice", "nunique")
).sort_values(
    "Total_Sales",
    ascending=False
)

country_analysis.head(10)

,Total_Sales,Total_Quantity,Total_Invoices
Country,,,
United Kingdom,8161692.763,4472606,23705
EIRE,352235.680,188659,457
Netherlands,263861.760,181822,150
Germany,195920.421,107026,513
France,130661.550,74317,297
Sweden,51190.110,52226,83
Denmark,46972.950,227030,33
Switzerland,43343.410,22053,50
Spain,37052.130,18317,85


### Country contribution

In [39]:
country_analysis["Sales_Contribution_%"] = (
    country_analysis["Total_Sales"]
    .div(country_analysis["Total_Sales"].sum())
    .mul(100)
)

country_analysis.head(10)

,Total_Sales,Total_Quantity,Total_Invoices,Sales_Contribution_%
Country,,,,
United Kingdom,8161692.763,4472606,23705,85.860356
EIRE,352235.680,188659,457,3.705491
Netherlands,263861.760,181822,150,2.775805
Germany,195920.421,107026,513,2.061067
France,130661.550,74317,297,1.374549
Sweden,51190.110,52226,83,0.538516
Denmark,46972.950,227030,33,0.494152
Switzerland,43343.410,22053,50,0.455969
Spain,37052.130,18317,85,0.389785


### Cancellation analysis

Invoices beginning with "C" represent cancellations in this dataset.

In [36]:
df["Is_Cancelled"] = df["Invoice"].astype(str).str.startswith("C")

In [37]:
cancellation_summary = df.groupby("Is_Cancelled").agg(
    Transactions=("Invoice", "count"),
    Sales=("Sales", "sum")
)

cancellation_summary

,Transactions,Sales
Is_Cancelled,,
False,505486,1.013521e+07
True,10182,-6.294351e+05


### Cancellation rate by country

In [38]:
country_cancellation = df.groupby("Country").agg(
    Total_Transactions=("Invoice", "count"),
    Cancelled_Transactions=("Is_Cancelled", "sum")
)

country_cancellation["Cancellation_Rate_%"] = (
    country_cancellation["Cancelled_Transactions"]
    .div(country_cancellation["Total_Transactions"])
    .mul(100)
)

country_cancellation.sort_values(
    "Cancellation_Rate_%",
    ascending=False
).head(10)

,Total_Transactions,Cancelled_Transactions,Cancellation_Rate_%
Country,,,
Japan,224,60,26.785714
Korea,63,10,15.873016
Channel Islands,906,85,9.381898
Lebanon,13,1,7.692308
United Arab Emirates,432,33,7.638889
Nigeria,32,2,6.250000
Poland,194,12,6.185567
Germany,8120,468,5.763547
USA,244,14,5.737705



## Business Insights

1. The United Kingdom is the dominant revenue market.
2. Customer 18102 generated the highest customer-level sales.
3. REGENCY CAKESTAND 3 TIER generated the highest product-level sales.
4. Cancellation behavior varies substantially across countries.
5. Monthly sales analysis using shift() enables period-over-period performance tracking.